In this code we will look at the gate we obtain from 'nn_tutorial.ipynb' and compute its fidelity using qutip. This is a nice sanity check, and we can also use qutip later to e.g. analyse the performance of the gates in presence of fluctuations on the controls as well as decay. To start off we can compute the Bell state fidelity with the $CZ$ gate. (Note to self: look for a nice test for the CPHASE gates)

In [1]:
# can start off importing 
import os 

os.chdir("/Users/mmohan/Library/CloudStorage/OneDrive-TUEindhoven/Pasqal_/parametrized_gates/time_optimal_phase_og/")

import qutip as qt 
import numpy as np 
import torch 
import cst_n_fn as cfn 
from schsolve import schsolver, neural_trainer 
import torchdiffeq as tdf 
import const_czphi as czphi 

device = 'cpu'

# file handling section start 
# we should also temporarily set the variable angle_batch in const_czphi.py to 1
# indeed test using cz_762_optimal to see if F = 1 for my code == F = 1 for qutip 

#dict_ = torch.load('data/rand_models/var_gatetime_cz_converged_2')

dict_ = torch.load('data/final_models/pt5pi_to_pi')
network = dict_['network']
print(network)
angle = torch.pi 
#gatetime = 0.47655
gatetime = network.gatetime_prediction.item()
print(gatetime*cfn.rabi)
time_arr = torch.linspace(0, gatetime, czphi.time_steps).reshape((czphi.time_steps, 1))

input_tensor = torch.tensor(angle).reshape(1,1)
init_matrix = torch.eye(3**2, dtype = torch.cfloat, requires_grad=True).unsqueeze(0)

pred_outputs_det = ((network(input_tensor).select(1, 0))*\
        (network.range_detuning[1] - network.range_detuning[0]) + network.range_detuning[0])


/opt/homebrew/Caskroom/miniconda/base/envs/baseClone/lib/python3.11/site-packages/qutip/__init__.py:65: UserWarning: The new version of Cython, (>= 3.0.0) is not supported.
  warnings.warn(


neural_trainer_time_optimal_cz(
  (ansatz_time): Sequential(
    (0): Linear(in_features=1, out_features=45, bias=True)
    (1): ReLU()
    (2): Linear(in_features=45, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (ansatz_control): Sequential(
    (0): Linear(in_features=2, out_features=300, bias=True)
    (1): ReLU()
    (2): Linear(in_features=300, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=300, out_features=300, bias=True)
    (5): ReLU()
    (6): Linear(in_features=300, out_features=300, bias=True)
    (7): ReLU()
    (8): Linear(in_features=300, out_features=300, bias=True)
    (9): ReLU()
    (10): Linear(in_features=300, out_features=300, bias=True)
    (11): ReLU()
    (12): Linear(in_features=300, out_features=300, bias=True)
    (13): ReLU()
    (14): Linear(in_features=300, out_features=300, bias=True)
    (15): ReLU()
    (16): Linear(in_features=300, out_features=300, bias=True)
    (17): ReLU()
    (18): Linear(in_features=300, out_feat

In [2]:
#rabi_arr = pred_outputs_rabi # so for pi 
#det_arr = pred_outputs_det

gatetime = torch.tensor(gatetime).reshape(1,1)
rabi_arr = cfn.const_then_zero_tensor(cfn.rabi, gatetime)
det_arr = czphi.list_to_fn_tensor_var_gatetime(pred_outputs_det, gatetime, czphi.time_steps)
    
czphi.instance.hamiltonian.rabi_tensored["pulse 0"] = \
        czphi.instance.hamiltonian.rabi_tensored["pulse 1"] = rabi_arr
czphi.instance.hamiltonian.det_tensored["pulse 0"] = \
        czphi.instance.hamiltonian.det_tensored["pulse 1"] = det_arr  
    
    # ! tasks: still need to solve multiple ODEs instead of one, natively     
sol_intrm = czphi.reduce_r_dim_2q_vector(tdf.odeint(czphi.instance,\
                init_matrix, torch.linspace(0, gatetime.item(), czphi.time_steps).to(device), 
                method = 'dopri5')[-1])

phi_01 = torch.angle(sol_intrm[:, 1, 1]).item()

#phi_01 = 2.1788077354431152

one_r = qt.basis(3, 1)*qt.basis(3, 2).dag()

h_1q_rabi = 0.5*(one_r + one_r.dag()) # * factor of 0.5 for the rabi frequency 

h_2q_rabi = qt.tensor(h_1q_rabi, qt.qeye(3)) + qt.tensor(qt.qeye(3), h_1q_rabi)

h_1q_det = qt.basis(3, 2)*qt.basis(3, 2).dag() # * detuning on the 1 - r transition 

h_2q_det = qt.tensor(h_1q_det, qt.qeye(3)) + qt.tensor(qt.qeye(3), h_1q_det)

# * interaction hamiltonian below

v_int = 21.1*cfn.rabi
rr = qt.tensor(qt.basis(3, 2), qt.basis(3, 2))
h_2q_int = rr*rr.dag()
# ? is there also a minus term for the detuning? 
# first we prepare a double Hadamard gate 
init_ = (1/np.sqrt(2))*(qt.basis(3, 0) + qt.basis(3, 1))
init_ = qt.tensor(init_, init_)
# * 
hamiltonian = [[h_2q_det, pred_outputs_det.detach().numpy()], v_int*h_2q_int + h_2q_rabi*cfn.rabi]
print(pred_outputs_det.detach().numpy().shape)
soln = qt.mesolve(hamiltonian, init_, np.linspace(0, gatetime.item(), czphi.time_steps))
print(czphi.time_steps)
soln = soln.states[-1].eliminate_states([2,5,6,7,8]) # * remove the rydberg states 
soln.dims = [[2,2], [1]]

# basically using torch.angle(soln[1,1] or soln[2,2]) -- as pulse is symmetric, phi_01 = phi_10 

oneq_rot = qt.Qobj(cfn.corr_1q_rotation(torch.tensor(phi_01)).detach().numpy())
oneq_rot.dims = [[2,2], [2,2]]
soln = oneq_rot*soln 
hadamard = qt.Qobj(((1/np.sqrt(2))*np.array([[1, 1],[1, -1]])))
had_op = qt.tensor(qt.qeye(2), hadamard)
soln = had_op*soln 
print("Fidelity of bell state preparation: ")
zer = qt.basis(2, 0)
one = qt.basis(2, 1)
bell = (1/np.sqrt(2))*(qt.tensor(zer, zer) + qt.tensor(one, one))
print(qt.fidelity(bell, soln)**2)



(201,)
201
Fidelity of bell state preparation: 
0.9997315893407885


In [3]:
time_arr = torch.linspace(0, czphi.gatetime, czphi.time_steps).reshape((czphi.time_steps, 1))

desired_angle = torch.FloatTensor(czphi.angle_batch, 1).uniform_(czphi.angle_domain[0], czphi.angle_domain[1]).to(device)
ideal_stack_epoch = cfn.czp_gate_stack(desired_angle)
desired_angle_expanded = desired_angle.repeat(1, czphi.time_steps).view(-1, 1)
input_tensor = torch.cat([desired_angle_expanded, time_arr], dim=-1)


AttributeError: module 'const_czphi' has no attribute 'gatetime'